<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%208/2_st445_week8_lab2_eda_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 8 · Lab 2: the EDA challenge

**A team game · about 100 minutes · practises the Week 7 EDA protocol and the Week 8 matrix methods on one dataset**

Two teams, **Team A** and **Team B**, answer questions about the same dataset using exploratory data analysis. After each round, each team **evaluates the other team's answer**: could they understand it clearly, vaguely, or not at all? Points go to the team whose answer was understood. The team with more points at the end wins.

### How a round works (10 minutes)

| Time | What happens |
|---|---|
| 6 minutes | Each team answers **its own question card** (Team A answers card A, Team B card B): code, one chart, and an insight statement |
| 2 minutes | Each team's **presenter** shows the chart and reads the insight statement to the other team (in person, or by sharing the screen) |
| 2 minutes | Each team **scores the other team's answer** with the rubric below, and records the score and a one-line reason |

The two cards in each round practise the **same skill** on different variables, so the round is fair and reduces the likelyhood of teams copying each other. There are **8 rounds** (40 minutes per four rounds, with a 10-minute break between rounds 4 and 5), then a **final question** for both teams, scored by the tutor as a tie-break.

### Scoring rubric: use it for every evaluation

| Points | The other team's answer was ... | In practice |
|---|---|---|
| **1** | understood **clearly** | it answers the question; it states an **observation**, gives specific **evidence** (the chart, with at least one number), and states a **caveat**; the chart is readable (title, labelled axes with units, suitable colours) |
| **0.5** | understood **vaguely** | it answers the question, but the evidence has no number, or the caveat is missing, or the chart is hard to read |
| **0** | **not understood** | it does not answer the question, or it is wrong, or there is no chart or no statement |

**Fair play.** Score the **clarity of the explanation**, not whether you agree with it. Every score needs a one-line reason; a team that disagrees with a score may appeal to the tutor, whose decision is final.

### Roles: rotate them every round

* **Analyst:** writes the code.
* **Writer:** writes the insight statement (observation, evidence, caveat).
* **Presenter:** presents the answer, and leads the scoring of the other team's answer.

With the above three roles per team, swap responsabilities so that everyone takes every role at least once.

**Before you start:** choose **File → Save a copy in Drive**, and fill in your team details in section 1.

---
## 1. Your team

In [ ]:
import numpy as np

# 🟢 Fill in, then run
MY_TEAM = "A"                                       # "A" or "B"
TEAM_NAME = "____"                                  # a name for the leaderboard
MEMBERS = ["____", "____", "____", "____", "____"]  # 3, 4 or 5 members

OTHER_TEAM = "B" if MY_TEAM == "A" else "A"
ROLES = ["analyst", "writer", "presenter"]

for r in range(1, 9):
    # Distribute members into roles. If there are 4 or 5 members, some roles will have multiple people
    n_members = len(MEMBERS)
    rotation = {role: [] for role in ROLES}

    # Assign each member to a role based on the round offset
    for i, member in enumerate(MEMBERS):
        role_index = (i - (r - 1)) % len(ROLES)
        rotation[ROLES[role_index]].append(member)

    # Clean up display: if a role has only 1 person, unpack it from list
    rotation_display = {role: names[0] if len(names) == 1 else names for role, names in rotation.items()}
    print(f"round {r}: {rotation_display}")

### Imports, style and the scoring tool

The cell below also defines `award(round_number, points, reason)`, which records the score **your team gives the other team** after each round, and `scoreboard()`, which shows the totals.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, optimal_leaf_ordering, leaves_list, fcluster
from scipy.spatial.distance import squareform, pdist
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import kneighbors_graph

OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]
ORIGIN_COLOURS = {"usa": OKABE_ITO[0], "europe": OKABE_ITO[1], "japan": OKABE_ITO[2]}
sns.set_theme(style="ticks")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SCORES_GIVEN = {}                              # round -> {"points": ..., "reason": ...}

def award(round_number, points, reason):
    """Record the score your team gives the OTHER team's answer in a round."""
    assert points in (0, 0.5, 1), "points must be 0, 0.5 or 1 (see the rubric)"
    assert len(reason.strip()) >= 10, "give a one-line reason for the score"
    SCORES_GIVEN[round_number] = {"points": points, "reason": reason.strip()}
    print(f"Round {round_number}: Team {OTHER_TEAM} receives {points} point(s). Reason: {reason.strip()}")

def scoreboard():
    """The scores your team has given, as a table, and their total."""
    table = pd.DataFrame(SCORES_GIVEN).T.rename_axis("round")
    total = table["points"].sum() if len(table) else 0
    print(f"Points given to Team {OTHER_TEAM} so far: {total}")
    return table

---
## 2. The dataset: Auto MPG

Fuel consumption and design of **398 cars** sold in the United States between **1970 and 1982**: fuel economy (miles per US gallon, **mpg**: higher is better), number of cylinders, engine displacement (cubic inches), horsepower, weight (pounds), acceleration (seconds from 0 to 60 mph: **lower is faster**), model year, origin (USA, Europe or Japan) and name.

* **Source:** Quinlan, J. R. (1993), "Combining instance-based and model-based learning", *Proceedings of the Tenth International Conference on Machine Learning*, 236–243; [UCI Machine Learning Repository, dataset 9](https://archive.ics.uci.edu/dataset/9/auto+mpg), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Loaded with [`sns.load_dataset("mpg")`](https://seaborn.pydata.org/generated/seaborn.load_dataset.html).
* **Why this dataset:** it needs almost no preprocessing, has a time dimension (model year), groups (origin, cylinders), a few genuinely missing values, a famous outlier, and seven numeric variables for matrix displays.

### Preprocessing, in one cell

In [ ]:
# 🟢 Load and prepare the data: one documented cell
raw = sns.load_dataset("mpg")                 # downloads a copy of the UCI file (internet needed)
cars = raw.copy()                             # keep raw unchanged

# 1. The real model year, instead of two digits (70 -> 1970)
cars["year"] = 1900 + cars["model_year"]
cars = cars.drop(columns="model_year")

# 2. Origin as a category with a fixed order, so every chart uses the same order and colours
cars["origin"] = pd.Categorical(cars["origin"], categories=["usa", "europe", "japan"])

# 3. The make (brand), the first word of the name, with the dataset's misspellings corrected
cars["make"] = (cars["name"].str.split().str[0]
                .replace({"chevy": "chevrolet", "chevroelt": "chevrolet", "vw": "volkswagen",
                          "vokswagen": "volkswagen", "toyouta": "toyota", "maxda": "mazda",
                          "mercedes-benz": "mercedes"}))

# 4. Horsepower is missing for 6 cars. We do NOT impute it here: round 2 asks you to examine
#    the gaps. 'complete' keeps only cars with every value, for the matrix methods (rounds 6 to 8)
NUMERIC = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "year"]
complete = cars.dropna(subset=NUMERIC).reset_index(drop=True)

# 5. A standardised copy (mean 0, sd 1) for distances between cars, so that no variable
#    dominates because of its units (weight in thousands of pounds would swamp everything else)
Z = (complete[NUMERIC] - complete[NUMERIC].mean()) / complete[NUMERIC].std()

# 6. Checks that fail loudly, that is, raise an error and stop the code from
# running.
assert cars.shape[0] == 398, "unexpected number of cars"
assert complete[NUMERIC].notna().all().all() and len(complete) == 392, "unexpected missing values"

print("all cars:", cars.shape, "| complete cars:", complete.shape)
cars.head()

**What you should see:** 398 cars in `cars` and 392 in `complete` (the 6 without horsepower removed). Use **`cars`** for Week 7 questions and **`complete`** and **`Z`** for the matrix questions.

### Your toolkit: helper functions from the Week 8 lecture

These are ready to use, so that you can spend the rounds on analysis rather than on writing algorithms.

In [ ]:
# 🟢 Week 8 helpers
def ar_events(D):
    """Gradient-condition violations (anti-Robinson events) of a dissimilarity matrix in its current order."""
    D = np.asarray(D)
    n = len(D)
    count = 0
    for i in range(n):
        for j in range(i + 2, n):
            k = np.arange(i + 1, j)
            count += int((D[i, k] > D[i, j]).sum()) + int((D[k, j] > D[i, j]).sum())
    return count

def seriate(D, method="average"):
    """An order for a dissimilarity matrix: hierarchical clustering + optimal leaf ordering."""
    condensed = squareform(np.asarray(D), checks=False)
    tree = optimal_leaf_ordering(linkage(condensed, method), condensed)
    return leaves_list(tree)

def fiedler_order(X, n_neighbors=10):
    """Spectral seriation: the Fiedler vector of a nearest-neighbour graph, and the order it gives."""
    A = kneighbors_graph(X, n_neighbors=n_neighbors, mode="connectivity").toarray()
    S = (A + A.T) / 2
    L = np.diag(S.sum(axis=1)) - S
    values, vectors = np.linalg.eigh(L)
    fiedler = vectors[:, 1]
    return fiedler, np.argsort(fiedler)

print("toolkit ready: ar_events(D), seriate(D), fiedler_order(X)")

### How to write each answer

For every card: one code cell that produces **one clear chart** (and any numbers you need), then the Markdown answer cell, completed:

> **Observation:** what you found, in one sentence that answers the question.
> **Evidence:** the chart, and at least one number from your output.
> **Caveat:** one thing that could make the observation wrong, or narrower.

Week 7's checklist applies to the chart: a title that states the finding, labelled axes with units, a suitable colour scale, and bars that start at zero.

---
## Round 1 · Week 7 · step 2: univariate distributions

**Hint for both cards:** `sns.histplot`, with a reference line at the median (`ax.axvline`); `.describe()` and `.skew()` for the numbers.

### Card A (Team A answers)

**Question.** How is **fuel economy (`mpg`)** distributed across the 398 cars? What is a typical value, how widely does it vary, and is the distribution symmetric?

In [ ]:
# Team A: your code for round 1

✍️ **Our answer (Team A, round 1)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** How is **horsepower** distributed across the cars? What is a typical value, how widely does it vary, and is the distribution symmetric?

In [ ]:
# Team B: your code for round 1

✍️ **Our answer (Team B, round 1)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`. Note that is requires three arguments: 1) round number, 2) your score to the OTHER team's answer, and 3) reason for your score.
For example:

`award(1, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 1 (0, 0.5 or 1), with a one-line reason. For example:
# award(1, 1, "clear observation; evidence with numbers; caveat about ...")

---
## Round 2 · Week 7 · steps 3 and 7: missingness and anomalies

**Hint for both cards:** A: `cars.isna().sum()`, then look at the rows with missing horsepower (origin, year). B: a scatter of acceleration against horsepower, and label the points furthest from the trend (`np.polyfit` or by eye).

### Card A (Team A answers)

**Question.** Which values are **missing** in the dataset, for which cars, and could the gaps **bias** an analysis of horsepower?

In [ ]:
# Team A: your code for round 2

✍️ **Our answer (Team A, round 2)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** Which cars are **unusual**: much slower or faster to accelerate than their **horsepower** would suggest? Are they errors or genuine?

In [ ]:
# Team B: your code for round 2

✍️ **Our answer (Team B, round 2)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`.

`award(2, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 2 (0, 0.5 or 1), with a one-line reason
# award(2, 1, "clear observation; evidence with numbers; caveat about ...")

---
## Round 3 · Week 7 · step 4: relationships

**Hint for both cards:** A scatter plot first (`sns.regplot` with `lowess=True` or `order=2` shows the shape), then a correlation (`.corr()`). Remember Anscombe.

### Card A (Team A answers)

**Question.** How are **weight** and **fuel economy** related? Describe the direction, the strength and the shape.

In [ ]:
# Team A: your code for round 3

✍️ **Our answer (Team A, round 3)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** How are **horsepower** and **acceleration** (0–60 mph time) related? Describe the direction, the strength and the shape.

In [ ]:
# Team B: your code for round 3

✍️ **Our answer (Team B, round 3)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`.

`award(3, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 3 (0, 0.5 or 1), with a one-line reason
# award(3, 1, "clear observation; evidence with numbers; caveat about ...")

---
## Round 4 · Week 7 · step 5: groups and subgroups

**Hint for both cards:** A: `pd.cut` to make weight bands, then a box plot or grouped bar of mpg by band and origin (Simpson's paradox from the lecture). B: a box plot of mpg by cylinders, with the group sizes shown.

### Card A (Team A answers)

**Question.** Japanese cars have the best fuel economy overall. Is that **only because they are lighter**? Compare origins **within** weight bands.

In [ ]:
# Team A: your code for round 4

✍️ **Our answer (Team A, round 4)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** How does fuel economy differ by **number of cylinders**? Are any groups **too small** to trust?

In [ ]:
# Team B: your code for round 4

✍️ **Our answer (Team B, round 4)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`.

`award(4, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 4 (0, 0.5 or 1), with a one-line reason
# award(4, 1, "clear observation; evidence with numbers; caveat about ...")

---
## ☕ Break: 10 minutes

Check the scoreboard below, and swap roles for round 5.

In [ ]:
scoreboard()

---
## Round 5 · Week 7 · step 6: time structure

**Hint for both cards:** Median by year (`groupby('year')`), drawn as a line; for B, one line per origin with direct labels. Start the y-axis at zero for honest comparisons of size.

### Card A (Team A answers)

**Question.** How did typical **fuel economy** change from 1970 to 1982? Was the change steady?

In [ ]:
# Team A: your code for round 5

✍️ **Our answer (Team A, round 5)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** How did typical **weight** change from 1970 to 1982, and did **every origin** follow the same path?

In [ ]:
# Team B: your code for round 5

✍️ **Our answer (Team B, round 5)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`.

`award(5, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 5 (0, 0.5 or 1), with a one-line reason
# award(5, 1, "clear observation; evidence with numbers; caveat about ...")

---
## Round 6 · Week 8: correlation heatmaps, Robinson form and seriation

**Hint for both cards:** A: `corr = complete[NUMERIC].corr()`, `D = 1 - corr.values`, then `order = seriate(D)` and `ar_events(...)` before and after. B: `complete[NUMERIC].cov()`; a diverging map centred on zero for correlation.

### Card A (Team A answers)

**Question.** Draw the **correlation heatmap** of the 7 numeric variables (use `complete`), first in the file's order, then **seriated**. Which groups of variables appear, and how much does seriation improve the order (gradient-condition violations before and after)?

In [ ]:
# Team A: your code for round 6

✍️ **Our answer (Team A, round 6)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** Draw the **covariance** and the **correlation** heatmaps of the same 7 variables side by side (use `complete`). Why would the covariance heatmap mislead a reader?

In [ ]:
# Team B: your code for round 6

✍️ **Our answer (Team B, round 6)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`.

`award(6, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 6 (0, 0.5 or 1), with a one-line reason
# award(6, 1, "clear observation; evidence with numbers; caveat about ...")

---
## Round 7 · Week 8: clustered heatmaps and k-means

**Hint for both cards:** A: `sns.clustermap(Z, method='ward', row_colors=...)`, then `fcluster(..., 3, 'maxclust')` and `pd.crosstab` with origin. B: `silhouette_score` for k = 2 to 7; `KMeans(n_clusters=k, n_init=10, random_state=445)`; sort the distance matrix by the labels.

### Card A (Team A answers)

**Question.** Draw a **clustered heatmap** of the cars × standardised variables (`Z`), with a strip showing each car's **origin** (added only as a check). Into how many main groups do the cars fall, and do they match origin?

In [ ]:
# Team A: your code for round 7

✍️ **Our answer (Team A, round 7)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** Use **k-means** on `Z`: choose k with the **silhouette**, then **reorder the distance matrix** by cluster. Describe the clusters (for example by cylinders and mpg).

In [ ]:
# Team B: your code for round 7

✍️ **Our answer (Team B, round 7)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`.

`award(7, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 7 (0, 0.5 or 1), with a one-line reason
# award(7, 1, "clear observation; evidence with numbers; caveat about ...")

---
## Round 8 · Week 8: spectral seriation

**Hint for both cards:** Correlate the Fiedler vector with each variable (`np.corrcoef`), then plot that variable in spectral order. B: plot year along the order, coloured by origin.

### Card A (Team A answers)

**Question.** Use **spectral seriation** (`fiedler_order(Z.values)`) to order the cars. Which **variable** does the gradient follow most closely? Show it.

In [ ]:
# Team A: your code for round 8

✍️ **Our answer (Team A, round 8)**

**Observation:**

**Evidence:**

**Caveat:**

### Card B (Team B answers)

**Question.** Use **spectral seriation** to order the cars. How do **model year** and **origin** vary along the spectral order? Show it.

In [ ]:
# Team B: your code for round 8

✍️ **Our answer (Team B, round 8)**

**Observation:**

**Evidence:**

**Caveat:**

✍️ **Evaluate the answer**

Now, give a score to the other team's answer using `award()`.

`award(8, 1, "clear observation; evidence with numbers; caveat about ...")`

In [ ]:
# Score the OTHER team's answer for round 8 (0, 0.5 or 1), with a one-line reason
# award(8, 1, "clear observation; evidence with numbers; caveat about ...")

---
## Final question (both teams): the tie-break

**Question.** Write **one** insight statement for a **non-specialist** reader (a car buyer in 1982): *what made a car fuel-efficient in these data?* Combine at least one Week 7 finding and one Week 8 (matrix) finding, with one chart.

The **tutor** scores both answers from 0 to 2: 2 for a clear, accurate statement a non-specialist could act on, with evidence and a caveat; 1 for a correct but technical or incomplete one; 0 otherwise. This score breaks a tie.

In [ ]:
# Your code for the final question

✍️ **Our final answer**

**Observation:**

**Evidence:**

**Caveat:**

---
## Results

Run the cell below to see the scores your team gave the other team, and save them to a file. **Send the file to the tutor** (via email or MS teams): the tutor adds the two teams' files to find the winner.

In [ ]:
board = scoreboard()
with open(f"scores_given_by_team_{MY_TEAM}.json", "w") as f:
    json.dump({"team": MY_TEAM, "team_name": TEAM_NAME, "scores_given_to": OTHER_TEAM, "scores": SCORES_GIVEN}, f, indent=2)
print(f"saved scores_given_by_team_{MY_TEAM}.json")
board

### References

* Quinlan, J. R. (1993). Combining instance-based and model-based learning. *Proceedings of the Tenth International Conference on Machine Learning*, 236–243. Data: UCI Machine Learning Repository, Auto MPG, [doi:10.24432/C5859H](https://doi.org/10.24432/C5859H).
* The Week 7 lecture (the EDA protocol; observation, evidence, caveat) and the Week 8 lecture (seriation, clustering, spectral methods); the ST445 visualisation checklist (Appendix B).